In [1]:
import numpy as np
import pandas as pd

distance_matrix = np.load('../data/distance_matrix.npy').astype(int)
sample = pd.read_csv('../data/sample_stops.csv')
routes_df = pd.read_csv('../data/solution_routes.csv')

N_VEHICLES = 3
CAPACITY = 25
DEPOT = 0

demand = sample['demand'].to_numpy()
print(f"{len(sample)-1} stops, {demand.sum()} packages, "
      f"{N_VEHICLES} vehicles x {CAPACITY} = {N_VEHICLES*CAPACITY} capacity")

20 stops, 66 packages, 3 vehicles x 25 = 75 capacity


In [2]:
def route_distance(seq):
    """Total meters for one route. seq must start and end at the depot."""
    return sum(distance_matrix[seq[i]][seq[i+1]] for i in range(len(seq) - 1))

def fleet_distance(routes):
    """Total meters across all routes, in km."""
    return sum(route_distance(r) for r in routes) / 1000

In [3]:
def baseline_dataset_order():
    """Stops in county-file order, packed into trucks until capacity is hit."""
    routes, current, load = [], [DEPOT], 0

    for node in range(1, len(sample)):
        d = int(demand[node])
        if load + d > CAPACITY:
            routes.append(current + [DEPOT])
            current, load = [DEPOT], 0
        current.append(node)
        load += d

    routes.append(current + [DEPOT])
    return routes

routes_naive = baseline_dataset_order()
print(f"drivers used: {len(routes_naive)}")
print(f"total: {fleet_distance(routes_naive):.2f} km")

drivers used: 3
total: 280.42 km


In [4]:
def baseline_nearest_neighbor():
    """From each position, drive to the closest stop that still fits."""
    unvisited = set(range(1, len(sample)))
    routes = []

    for _ in range(N_VEHICLES):
        current, load, position = [DEPOT], 0, DEPOT

        while True:
            feasible = [n for n in unvisited if load + demand[n] <= CAPACITY]
            if not feasible:
                break
            nxt = min(feasible, key=lambda n: distance_matrix[position][n])
            current.append(nxt)
            load += int(demand[nxt])
            unvisited.remove(nxt)
            position = nxt

        routes.append(current + [DEPOT])

    if unvisited:
        print(f"WARNING: {len(unvisited)} stops unserved by nearest neighbor")
    return routes

routes_nn = baseline_nearest_neighbor()
print(f"drivers used: {len(routes_nn)}")
print(f"total: {fleet_distance(routes_nn):.2f} km")

drivers used: 3
total: 156.91 km
